# **Plant Disease Detection**

In [ ]:
# ── CELL 1: Upload Kaggle API Key & Download Dataset ─────────
# Heading: 1. Download and Install Dataset
# Link: https://www.kaggle.com/datasets/emmarex/plantdisease

from google.colab import files
files.upload()   # Upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d emmarex/plantdisease
!unzip plantdisease.zip -d plant_disease

In [ ]:
# ── CELL 2: Check Folder Structure ───────────────────────────
# Heading: Check Dataset Folder

!ls plant_disease/
!ls plant_disease/plantvillage/

In [ ]:
# ── CELL 3: Split Dataset into Train/Test (80/20) ────────────
# Heading: Step: Split Dataset into Train and Test Folders

import os
import shutil
from sklearn.model_selection import train_test_split

# Source path (your original dataset)
source_dir = "plant_disease/plantvillage/PlantVillage"

# Destination paths
train_dir = "plant_disease/train"
test_dir  = "plant_disease/test"

# Create destination folders
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Loop through each class folder
for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)
    if not os.path.isdir(class_path):
        continue  # skip files

    images = os.listdir(class_path)
    train_imgs, test_imgs = train_test_split(images, test_size=0.2, random_state=42)

    # Create class folders in train and test dirs
    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    # Copy training images
    for img in train_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(train_dir, class_name, img)
        shutil.copy2(src, dst)

    # Copy testing images
    for img in test_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(test_dir, class_name, img)
        shutil.copy2(src, dst)

print("Dataset successfully split into 80% train and 20% test.")
# This creates folder structure like:
# plant_disease/train/
# plant_disease/test/
# Each folder contains subfolders for different disease categories.

In [ ]:
# ── CELL 4: Data Preprocessing & Augmentation ────────────────
# Heading: 2. Data Preprocessing & Augmentation

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 128
BATCH_SIZE = 32

train_dir = 'plant_disease/train'
test_dir  = 'plant_disease/test'

train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=20
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)


In [ ]:
# ── CELL 5: Build the CNN Model ──────────────────────────────
# Heading: 3. Build the CNN Model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(train_generator.num_classes, activation='softmax')
])

In [ ]:
# ── CELL 6: Compile & Train the Model ────────────────────────
# Heading: 4. Compile & Train the Model

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=10
)

In [ ]:
# ── CELL 7: Evaluate the Model ───────────────────────────────
# Heading: 5. Evaluate the Model

loss, acc = model.evaluate(test_generator)
print(f"Test Accuracy: {acc*100:.2f}%")

In [ ]:
# ── CELL 8: Predict on a New Image ───────────────────────────
# Heading: 6. Predict on a New Image

import numpy as np
from tensorflow.keras.preprocessing import image

# Upload your own leaf image to test
from google.colab import files
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

img = image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
predicted_class = train_generator.class_indices
predicted_label = list(predicted_class.keys())[np.argmax(prediction)]

print(f"Predicted Disease: {predicted_label}")

In [ ]:
# ── CELL 9: Visualize Training ───────────────────────────────
# Heading: 7. Visualize Training

import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title("Accuracy over Epochs")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()